In [ ]:
# ── 1. Imports ──────────────────────────────────────────────────────────────
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

from sklearn.ensemble import IsolationForest, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    f1_score, precision_score, recall_score,
)
import xgboost as xgb

sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['figure.dpi'] = 120
RANDOM_STATE = 42

print('✅  Imports OK')

## 2. Dataset Generation
We generate synthetic but realistic bus operation data that mirrors the statistics of real DTC (Delhi Transport Corporation) routes.

In [ ]:
# ── 2. Synthetic Dataset ────────────────────────────────────────────────────
np.random.seed(RANDOM_STATE)
N = 5000

hour       = np.random.randint(5, 24, N)
dow        = np.random.randint(0,  7, N)
is_weekend = (dow >= 5).astype(int)
weather    = np.random.choice([0, 1, 2, 3], N, p=[0.5, 0.25, 0.15, 0.10])  # clear/cloudy/rain/fog
avg_temp   = np.random.normal(28, 7, N)
load_pct   = np.random.beta(2, 2, N) * 100  # 0–100 %
distance   = np.random.gamma(3, 4, N)        # km, right-skewed
stops      = np.random.randint(5, 25, N)
speed      = np.random.normal(28, 8, N).clip(5, 60)

# ── ETA ground truth ────────────────────────────────────────────────────────
base_eta   = (distance / speed) * 60
traffic    = np.where((hour >= 8) & (hour <= 10) | (hour >= 17) & (hour <= 20), np.random.normal(8, 3, N), np.random.normal(2, 1, N))
wx_penalty = weather * np.random.uniform(1, 4, N)
eta_true   = (base_eta + traffic.clip(0) + wx_penalty + np.random.normal(0, 2, N)).clip(2, 180)

# ── Delay ground truth ──────────────────────────────────────────────────────
delay_true = (traffic + wx_penalty + load_pct * 0.05 + np.random.normal(0, 3, N)).clip(0, 60)

df = pd.DataFrame({
    'hour': hour, 'dow': dow, 'is_weekend': is_weekend,
    'weather': weather, 'avg_temp': avg_temp, 'load_pct': load_pct,
    'distance': distance, 'stops': stops, 'speed': speed,
    'eta_true': eta_true, 'delay_true': delay_true,
})

print(f'Dataset shape: {df.shape}')
df.describe().round(2)

## 3. Anomaly Detection — Isolation Forest

In [ ]:
# ── 3.1 Train ───────────────────────────────────────────────────────────────
anomaly_features = df[['speed', 'delay_true', 'load_pct']].values
scaler_an = StandardScaler()
X_an      = scaler_an.fit_transform(anomaly_features)

iso = IsolationForest(n_estimators=200, contamination=0.05, random_state=RANDOM_STATE)
iso.fit(X_an)
scores   = iso.score_samples(X_an)
preds    = iso.predict(X_an)   # -1 = anomaly, 1 = normal
n_anom   = (preds == -1).sum()

print(f'Anomalies detected: {n_anom} / {N}  ({n_anom/N*100:.1f}%)')

In [ ]:
# ── 3.2 Visualise anomaly scores ───────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Score distribution
axes[0].hist(scores, bins=60, color='steelblue', edgecolor='none')
axes[0].axvline(iso.offset_, color='red', linestyle='--', label=f'Decision boundary ({iso.offset_:.3f})')
axes[0].set_title('Anomaly Score Distribution')
axes[0].set_xlabel('Score (more negative → more anomalous)')
axes[0].set_ylabel('Count')
axes[0].legend()

# Scatter: speed vs delay coloured by anomaly
colors = ['#EF4444' if p == -1 else '#3B82F6' for p in preds]
axes[1].scatter(df['speed'], df['delay_true'], c=colors, alpha=0.3, s=8)
axes[1].set_xlabel('Speed (km/h)')
axes[1].set_ylabel('Delay (min)')
axes[1].set_title('Speed vs Delay — Anomaly Overlay')
from matplotlib.lines import Line2D
axes[1].legend(handles=[
    Line2D([0],[0], marker='o', color='w', markerfacecolor='#EF4444', label='Anomaly'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor='#3B82F6', label='Normal'),
], markerscale=1.5)

plt.tight_layout()
plt.savefig('anomaly_evaluation.png', bbox_inches='tight')
plt.show()
print('📊  Saved: anomaly_evaluation.png')

## 4. ETA Prediction — Gradient Boosting Regressor

In [ ]:
# ── 4.1 Train / Eval split ──────────────────────────────────────────────────
eta_features = ['distance', 'hour', 'dow', 'is_weekend', 'weather', 'speed', 'load_pct', 'avg_temp']
X_eta  = df[eta_features].values
y_eta  = df['eta_true'].values

X_tr, X_te, y_tr, y_te = train_test_split(X_eta, y_eta, test_size=0.2, random_state=RANDOM_STATE)
scaler_eta = StandardScaler()
X_tr_s = scaler_eta.fit_transform(X_tr)
X_te_s = scaler_eta.transform(X_te)

gbr = GradientBoostingRegressor(n_estimators=300, max_depth=5, learning_rate=0.05, random_state=RANDOM_STATE)
gbr.fit(X_tr_s, y_tr)

y_pred_eta = gbr.predict(X_te_s)

mae  = mean_absolute_error(y_te, y_pred_eta)
rmse = mean_squared_error(y_te, y_pred_eta) ** 0.5
r2   = r2_score(y_te, y_pred_eta)
print(f'ETA Predictor — MAE: {mae:.2f} min | RMSE: {rmse:.2f} min | R²: {r2:.4f}')

In [ ]:
# ── 4.2 Visualise ──────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Actual vs Predicted
axes[0].scatter(y_te, y_pred_eta, alpha=0.3, s=8, color='#8B5CF6')
lim = [min(y_te.min(), y_pred_eta.min()), max(y_te.max(), y_pred_eta.max())]
axes[0].plot(lim, lim, 'r--', label='Perfect fit')
axes[0].set_xlabel('Actual ETA (min)')
axes[0].set_ylabel('Predicted ETA (min)')
axes[0].set_title(f'ETA: Actual vs Predicted  (R²={r2:.3f})')
axes[0].legend()

# Feature importance
imp = pd.Series(gbr.feature_importances_, index=eta_features).sort_values(ascending=True)
imp.plot(kind='barh', ax=axes[1], color='#3B82F6')
axes[1].set_title('Feature Importance — ETA Predictor')
axes[1].set_xlabel('Importance')

plt.tight_layout()
plt.savefig('eta_evaluation.png', bbox_inches='tight')
plt.show()
print('📊  Saved: eta_evaluation.png')

## 5. Delay Prediction — XGBoost

In [ ]:
# ── 5.1 Regression ─────────────────────────────────────────────────────────
delay_features = ['hour', 'dow', 'is_weekend', 'weather', 'avg_temp', 'load_pct', 'distance', 'stops']
X_d  = df[delay_features].values
y_d  = df['delay_true'].values

X_tr2, X_te2, y_tr2, y_te2 = train_test_split(X_d, y_d, test_size=0.2, random_state=RANDOM_STATE)

xgb_reg = xgb.XGBRegressor(n_estimators=400, max_depth=6, learning_rate=0.05,
                            subsample=0.8, colsample_bytree=0.8, random_state=RANDOM_STATE,
                            verbosity=0)
xgb_reg.fit(X_tr2, y_tr2, eval_set=[(X_te2, y_te2)], verbose=False)
y_pred_delay = xgb_reg.predict(X_te2)

mae_d  = mean_absolute_error(y_te2, y_pred_delay)
rmse_d = mean_squared_error(y_te2, y_pred_delay) ** 0.5
r2_d   = r2_score(y_te2, y_pred_delay)
print(f'Delay Regressor — MAE: {mae_d:.2f} min | RMSE: {rmse_d:.2f} min | R²: {r2_d:.4f}')

# ── 5.2 Classification (on-time / moderate / severe) ───────────────────────
def delay_class(d):
    if d <= 5:  return 0   # on-time
    if d <= 15: return 1   # moderate
    return 2               # severe

y_cls = np.vectorize(delay_class)(y_d)
X_tr3, X_te3, y_tr3, y_te3 = train_test_split(X_d, y_cls, test_size=0.2, random_state=RANDOM_STATE)

xgb_cls = xgb.XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.05,
                             num_class=3, eval_metric='mlogloss',
                             random_state=RANDOM_STATE, verbosity=0)
xgb_cls.fit(X_tr3, y_tr3, eval_set=[(X_te3, y_te3)], verbose=False)
y_pred_cls = xgb_cls.predict(X_te3)

print('\nDelay Classifier Report:')
print(classification_report(y_te3, y_pred_cls, target_names=['On-Time', 'Moderate', 'Severe']))

In [ ]:
# ── 5.3 Confusion matrix + Feature importance ──────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm = confusion_matrix(y_te3, y_pred_cls)
ConfusionMatrixDisplay(cm, display_labels=['On-Time', 'Moderate', 'Severe']).plot(ax=axes[0], colorbar=False)
axes[0].set_title('XGBoost Delay Classifier — Confusion Matrix')

imp_d = pd.Series(xgb_cls.feature_importances_, index=delay_features).sort_values(ascending=True)
imp_d.plot(kind='barh', ax=axes[1], color='#F59E0B')
axes[1].set_title('Feature Importance — Delay Classifier')

plt.tight_layout()
plt.savefig('delay_evaluation.png', bbox_inches='tight')
plt.show()
print('📊  Saved: delay_evaluation.png')

## 6. Schedule Optimizer — Genetic Algorithm Convergence

In [ ]:
# ── 6. Run GA optimizer and track fitness evolution ────────────────────────
import sys, os
sys.path.insert(0, '..')

import importlib.util, types

# Minimal stub so optimizer can import predict_demand without the full AI stack
def _mock_predict(**kwargs):
    h = kwargs.get('hour', 8)
    return {'predicted_count': max(10, 80 * np.sin(np.pi * (h - 5) / 18) + np.random.randint(-5, 5))}

import unittest.mock as mock

with mock.patch.dict('sys.modules', {'predictors': mock.MagicMock(predict_demand=_mock_predict)}):
    spec = importlib.util.spec_from_file_location('optimizer', os.path.join('..', 'optimizer.py'))
    optimizer_mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(optimizer_mod)

# Patch internal demand call for pure eval
optimizer_mod._get_demand = lambda route_id, date, h, is_we, is_hol: max(10, int(80 * np.sin(np.pi * (h-5)/18)))

# Run with fitness tracking
best_per_gen = []
_orig_run = optimizer_mod.optimize_headway

result = optimizer_mod.optimize_headway(
    route_id='R1', date='2025-01-06',
    fleet_size=5, is_weekend=False, is_holiday=False,
    start_hour=5, end_hour=22,
    population_size=80, generations=150
)
print(f"Slots: {len(result['slots'])} | Total wait score: {result['total_wait_score']:.0f}")
print(f"Optimization info: {result['optimization_info']}")

In [ ]:
# ── 6.2 Visualise optimised schedule ────────────────────────────────────────
slots  = result['slots']
hours  = [s['hour']         for s in slots]
demand = [s['demand_score'] for s in slots]
crowd  = [s['crowd_level']  for s in slots]
color_map = {'low': '#10B981', 'medium': '#F59E0B', 'high': '#EF4444', 'critical': '#7C3AED'}
colors = [color_map.get(c, '#6B7280') for c in crowd]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(hours, demand, color=colors, width=0.6, edgecolor='none')
axes[0].set_title('GA-Optimised Schedule — Demand by Departure Hour')
axes[0].set_xlabel('Hour of Day')
axes[0].set_ylabel('Demand Score')
from matplotlib.patches import Patch
axes[0].legend(handles=[Patch(facecolor=c, label=l) for l, c in color_map.items()], title='Crowd Level')

# Departure timeline
dep_mins = [s['departure_min'] for s in slots]
axes[1].scatter(dep_mins, [1] * len(dep_mins), c=colors, s=120, zorder=3)
axes[1].set_xlim(dep_mins[0] - 15, dep_mins[-1] + 15)
axes[1].set_yticks([])
axes[1].set_xlabel('Minutes from midnight')
axes[1].set_title('Dispatch Timeline')
for m, dep in zip(dep_mins, [s['departure_time_str'] for s in slots]):
    if dep_mins.index(m) % 3 == 0:
        axes[1].annotate(dep, (m, 1), textcoords='offset points', xytext=(0, 10), ha='center', fontsize=8)

plt.tight_layout()
plt.savefig('schedule_evaluation.png', bbox_inches='tight')
plt.show()
print('📊  Saved: schedule_evaluation.png')

## 7. Model Performance Summary

In [ ]:
# ── 7. Summary table ────────────────────────────────────────────────────────
summary = pd.DataFrame([
    {'Model': 'Anomaly Detector (IsoForest)',  'Metric': 'Contamination',  'Value': f'{n_anom/N*100:.1f}%'},
    {'Model': 'ETA Predictor (GBR)',           'Metric': 'MAE (min)',       'Value': f'{mae:.2f}'},
    {'Model': 'ETA Predictor (GBR)',           'Metric': 'RMSE (min)',      'Value': f'{rmse:.2f}'},
    {'Model': 'ETA Predictor (GBR)',           'Metric': 'R²',              'Value': f'{r2:.4f}'},
    {'Model': 'Delay Regressor (XGBoost)',     'Metric': 'MAE (min)',       'Value': f'{mae_d:.2f}'},
    {'Model': 'Delay Regressor (XGBoost)',     'Metric': 'RMSE (min)',      'Value': f'{rmse_d:.2f}'},
    {'Model': 'Delay Regressor (XGBoost)',     'Metric': 'R²',              'Value': f'{r2_d:.4f}'},
    {'Model': 'Delay Classifier (XGBoost)',   'Metric': 'F1 (macro)',      'Value': f"{f1_score(y_te3, y_pred_cls, average='macro'):.4f}"},
    {'Model': 'Delay Classifier (XGBoost)',   'Metric': 'Precision (macro)', 'Value': f"{precision_score(y_te3, y_pred_cls, average='macro'):.4f}"},
    {'Model': 'Delay Classifier (XGBoost)',   'Metric': 'Recall (macro)',  'Value': f"{recall_score(y_te3, y_pred_cls, average='macro'):.4f}"},
    {'Model': 'GA Schedule Optimizer',        'Metric': 'Slots Generated', 'Value': str(len(slots))},
    {'Model': 'GA Schedule Optimizer',        'Metric': 'Total Wait Score','Value': f"{result['total_wait_score']:.0f}"},
])

print('\n' + '='*55)
print('        SMARTDTC AI MODEL EVALUATION SUMMARY')
print('='*55)
print(summary.to_string(index=False))